In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score
import os
import requests
import zipfile
import io
import gzip
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

In [ ]:
# Create directory and download EMNIST dataset
os.makedirs('res', exist_ok=True)
data = requests.get('https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip')
files = zipfile.ZipFile(io.BytesIO(data.content))
files.extractall('res')

# Validation split & batch size
validation_split = 0.3
batch_size = 512
num_of_classes = 62  # MNIST ByClass classes

# Function to read MNIST images
def read_MNIST_images(filename):
    with gzip.open(filename, 'rb') as file:
        images = np.frombuffer(file.read(), np.uint8, offset=16)
    return images.reshape(-1, 28, 28).astype("float32") / 255.0  # No need for (N, H, W, C) in PyTorch

# Function to read MNIST labels
def read_MNIST_labels(filename):
    with gzip.open(filename, 'rb') as file:
        labels = np.frombuffer(file.read(), np.uint8, offset=8)
    return labels

# Load data
x_train = read_MNIST_images('res/gzip/emnist-byclass-train-images-idx3-ubyte.gz')
y_train = read_MNIST_labels('res/gzip/emnist-byclass-train-labels-idx1-ubyte.gz')
x_test = read_MNIST_images('res/gzip/emnist-byclass-test-images-idx3-ubyte.gz')
y_test = read_MNIST_labels('res/gzip/emnist-byclass-test-labels-idx1-ubyte.gz')

# Split into train & validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=validation_split, random_state=42)

#------------------------------------------------
# Uncomment this part to test a smaller dataset
# dataset_sample_val = 0.1
# x_train, _, y_train, _ = train_test_split(x_train, y_train, test_size=(1 - dataset_sample_val), random_state=42)
# x_val, _, y_val, _ = train_test_split(x_val, y_val, test_size=(1 - dataset_sample_val), random_state=42)
# x_test, _, y_test, _ = train_test_split(x_test, y_test, test_size=(1 - dataset_sample_val), random_state=42)
#------------------------------------------------

# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)  # (N, H, W) → (N, 1, H, W)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

x_val_tensor = torch.tensor(x_val, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# One-hot encode labels (if needed)
y_train_tensor = F.one_hot(y_train_tensor, num_classes=num_of_classes).float()
y_val_tensor = F.one_hot(y_val_tensor, num_classes=num_of_classes).float()
y_test_tensor = F.one_hot(y_test_tensor, num_classes=num_of_classes).float()

# PyTorch Dataset class
class MNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Create datasets
train_dataset = MNISTDataset(x_train_tensor, y_train_tensor)
val_dataset = MNISTDataset(x_val_tensor, y_val_tensor)
test_dataset = MNISTDataset(x_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
train_dataset = ImageFolder(root='data/train', transform=train_transform)
val_dataset = ImageFolder(root='data/val', transform=val_test_transform)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_of_classes=10):
        super(CNNModel, self).__init__()
        
        self.conv1 = nn.Conv2d(16, 16, kernel_size=5, stride=1, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        
        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(16)
        
        self.dwconv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn3 = nn.BatchNorm2d(16)
        
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.15)
        
        self.dwconv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn5 = nn.BatchNorm2d(32)
        
        self.dwconv6 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn6 = nn.BatchNorm2d(32)
        
        self.conv7 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn7 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.2)
        
        self.dwconv8 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn8 = nn.BatchNorm2d(64)
        
        self.dwconv9 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn9 = nn.BatchNorm2d(64)
        
        self.conv10 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn10 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.2)
        
        self.dwconv11 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn11 = nn.BatchNorm2d(64)
        
        self.dwconv12 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn12 = nn.BatchNorm2d(64)
        
        self.conv13 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn13 = nn.BatchNorm2d(64)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop4 = nn.Dropout(0.2)
        
        self.dwconv14 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn14 = nn.BatchNorm2d(64)
        
        self.dwconv15 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn15 = nn.BatchNorm2d(64)
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.drop5 = nn.Dropout(0.25)
        self.fc = nn.Linear(64, num_of_classes)
        
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.dwconv2(x)))
        x = F.relu(self.bn3(self.dwconv3(x)))
        
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool1(x)
        x = self.drop1(x)
        
        x = F.relu(self.bn5(self.dwconv5(x)))
        x = F.relu(self.bn6(self.dwconv6(x)))
        
        x = F.relu(self.bn7(self.conv7(x)))
        x = self.pool2(x)
        x = self.drop2(x)
        
        x = F.relu(self.bn8(self.dwconv8(x)))
        x = F.relu(self.bn9(self.dwconv9(x)))
        
        x = F.relu(self.bn10(self.conv10(x)))
        x = self.pool3(x)
        x = self.drop3(x)
        
        x = F.relu(self.bn11(self.dwconv11(x)))
        x = F.relu(self.bn12(self.dwconv12(x)))
        
        x = F.relu(self.bn13(self.conv13(x)))
        x = self.pool4(x)
        x = self.drop4(x)
        
        x = F.relu(self.bn14(self.dwconv14(x)))
        x = F.relu(self.bn15(self.dwconv15(x)))
        
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.drop5(x)
        x = self.fc(x)
        return x

model = CNNModel(num_of_classes=10)
print(model)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=2e-3)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=2.5e-5)

# Early Stopping (Manual Implementation)
class EarlyStopping:
    def __init__(self, patience=10, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_weights = model.state_dict() if self.restore_best_weights else None
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                if self.restore_best_weights and self.best_model_weights is not None:
                    model.load_state_dict(self.best_model_weights)
                return True
        return False

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

# Training Loop
num_epochs = 60
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_dataset:
        inputs, labels = batch
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = torch.nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_dataset:
            inputs, labels = batch
            outputs = model(inputs)
            loss = torch.nn.CrossEntropyLoss()(outputs, labels)
            val_loss += loss.item()
    
    # Reduce LR on Plateau
    scheduler.step(val_loss)

    # Early Stopping Check
    if early_stopping(val_loss, model):
        break

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")